# آهنگ‌بان — تست واقعی لب‌خوانی با GPU رایگان

**فقط تست کوتاه: حدود ۶٫۲۸ ثانیه سجاد و ۴٫۴۹ ثانیه آوا.** صدای موجود حفظ می‌شود؛ صدای جدید یا کلون ساخته نمی‌شود. این نوت‌بوک هنوز روی GPU Colab آزمایش نشده؛ خطا یا کیفیت نامناسب ممکن است. نوت‌بوک، به‌تنهایی، نتیجهٔ موفق لب‌خوانی نیست.

### قبل از اجرا
۱. از **Runtime → Change runtime type** یک **GPU رایگان** انتخاب کن؛ اگر T4 رایگان در دسترس بود همان را بگذار. هیچ Pro / Compute Units / کارت پرداختی لازم نیست. کد نمی‌تواند طرح حساب یا صورت‌حساب گوگل را تشخیص بدهد؛ خودت حالت رایگان را انتخاب کن.
۲. دو تیک سلول اول را بزن و **Runtime → Run all** را اجرا کن. اگر GPU رایگان ندادی یا خطا گرفت، چیزی نخر؛ پیام خطا را برای بررسی بفرست.
۳. نصب اولیه و دانلود چند گیگابایت مدل زمان‌بر است. اگر همهٔ مراحل موفق شوند، دو ویدیو و ZIP آخر آماده می‌شوند؛ ZIP را به همین گفت‌وگو برگردان.

### داده‌ها و حریم خصوصی
با اجرا، **گوگل** تصویر فعلی استودیو/مجریان و **کل MP4 قبلی (~۱۷MB)** را از مخزن عمومی خودت دریافت می‌کند تا دو قطعهٔ کوتاه استخراج شوند. فایل خام `sajad.m4a` دریافت نمی‌شود. فقط همین دو قطعه وارد مدل می‌شوند. وزن‌های عمومی از GitHub/Hugging Face رسمی دانلود می‌شوند؛ فایل تو به Hugging Face یا سرویس تولید ویدیوی دیگری ارسال نمی‌شود. Drive متصل نمی‌شود و سرور عمومی/Gradio نداریم. هیچ فایل یا ویدیو خودکار در GitHub یا یوتیوب منتشر نمی‌شود.

پس از دانلود ZIP، **Runtime → Disconnect and delete runtime** را بزن؛ نگهداری داده تابع شرایط خود گوگل است. ZIP فقط ویدیوهای ترکیب‌شده و گزارش‌ها را دارد، نه WAV جدا، صورت ورودی یا وزن‌ها.


In [ ]:
#@title ۱) تأیید طرح رایگان و پردازش در گوگل + بررسی GPU
FREE_RUNTIME_CONFIRMED = False #@param {type:"boolean"}
GOOGLE_PROCESSING_CONFIRMED = False #@param {type:"boolean"}

PILOT_GPU_READY = SETUP_OK = INPUTS_OK = MODELS_OK = RESULTS_READY = False
if not FREE_RUNTIME_CONFIRMED or not GOOGLE_PROCESSING_CONFIRMED:
    raise RuntimeError("دو تیک بالا را فقط پس از انتخاب GPU رایگان و پذیرش پردازش فایل‌ها در گوگل بزن، سپس Run all را اجرا کن.")
import sys, os, json, shutil, subprocess, urllib.request, zipfile
from pathlib import Path
try:
    import google.colab
except ImportError:
    raise RuntimeError("این نوت‌بوک برای محیط Google Colab رایگان آماده شده است.")
if shutil.which('nvidia-smi') is None:
    raise RuntimeError("GPU رایگان تخصیص نیافته. Runtime → Change runtime type را بررسی کن؛ چیزی نخر.")
probe = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader,nounits'], capture_output=True, text=True)
if probe.returncode:
    raise RuntimeError("GPU قابل استفاده نیست؛ اجرا متوقف شد. حالت پولی جایگزین نمی‌شود.")
print(probe.stdout)
if int(probe.stdout.splitlines()[0].rsplit(',',1)[1].strip()) < 6*1024:
    raise RuntimeError("برای این پایلوت حداقل ۶GiB حافظه GPU لازم است؛ چیزی نخر.")
if shutil.disk_usage('/content').free < 20*1024**3:
    raise RuntimeError("حداقل ۲۰GiB فضای آزاد برای نصب/مدل لازم است؛ فضای جلسه را بررسی کن، خرید لازم نیست.")
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
os.environ['DO_NOT_TRACK']='1'
os.environ['HF_ENDPOINT']='https://huggingface.co'
ROOT=Path('/content/ahangban-pilot')
ENV=Path('/content/ahangban-py310')
PY=ENV/'bin/python'
PILOT_GPU_READY=True
print("بررسی اولیه پاس شد؛ این بررسی نوع اشتراک گوگل را تشخیص نمی‌دهد.")


## ۲) نصب جدا از پایتون اصلی Colab
محیط مستقل Python 3.10 و Torch/CUDA سازگار با MuseTalk ساخته می‌شود. از مدل تشخیص ژست/چهره استفاده نمی‌کنیم؛ صورت‌های ثابت خودمان جعبهٔ دستی دارند. هیچ TensorFlow، Gradio، MMCV یا MMPose نصب نمی‌شود.


In [ ]:
#@title ۲) دریافت منابع قفل‌شده و ساخت محیط (اولین بار زمان‌بر است)
assert PILOT_GPU_READY and FREE_RUNTIME_CONFIRMED and GOOGLE_PROCESSING_CONFIRMED
SETUP_OK = INPUTS_OK = MODELS_OK = RESULTS_READY = False
CODE_REVISION = "6876b49bfc2935bb3b16329883c4c13119a1a51d"
FILES = ['content/005-cinematic-presenters/colab/run_pilot.py', 'content/005-cinematic-presenters/colab/pilot-settings.json', 'content/005-cinematic-presenters/colab/requirements.txt', 'content/005-cinematic-presenters/colab/MuseTalk-NOTICE.txt', 'content/005-cinematic-presenters/prepare_lipsync_pilot.py', 'content/005-cinematic-presenters/build_camera_demo.py', 'content/003-chavoshi-studio/build_video.py', 'content/003-chavoshi-studio/timing.py', 'content/004-chavoshi-own-voice/delivery/timeline.json', 'downloads/ahangban-first-video-final.mp4', 'studio/ahangban-signature/productions/HOSTS-VIEW-002.png', 'studio/ahangban-signature/overlays/AHANGBAN-LOGO-CORE.png', 'studio/ahangban-signature/overlays/AHANGBAN-LOGO-GLOW.png']
BASE=f'https://raw.githubusercontent.com/Masoumiofficial/ahangban-youtube/{CODE_REVISION}/'
ROOT.mkdir(parents=True,exist_ok=True)
for relative in FILES:
    target=ROOT/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    partial=target.with_name(target.name+'.partial')
    print('دریافت:',relative,flush=True)
    urllib.request.urlretrieve(BASE+relative,partial)
    partial.replace(target)
COLAB_DIR=ROOT/'content/005-cinematic-presenters/colab'
RUNNER=COLAB_DIR/'run_pilot.py'
def run(command,**kwargs):
    subprocess.run([str(x) for x in command],check=True,**kwargs)
run([sys.executable,'-m','pip','install','uv==0.9.5'])
run([sys.executable,'-m','uv','python','install','3.10'])
if not PY.is_file():
    run([sys.executable,'-m','uv','venv','--python','3.10','--seed',ENV])
else:
    run([PY,'-c',"import sys; assert sys.version_info[:2]==(3,10), 'Unexpected existing environment'"])
run([PY,'-m','pip','install','torch==2.0.1','torchvision==0.15.2','--index-url','https://download.pytorch.org/whl/cu118'])
run([PY,'-m','pip','install','-r',COLAB_DIR/'requirements.txt'])
run([PY,'-m','pip','check'])
run([PY,'-c',"import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO CUDA'); assert torch.cuda.is_available(), 'GPU/CUDA unavailable: stop, no paid fallback'"])
SETUP_OK=True
print('محیط جدا آماده است؛ پایتون اصلی Colab عوض نشده است.')


In [ ]:
#@title ۳) آماده‌کردن قطعه‌های واقعی و نمایش محدودهٔ لب
assert SETUP_OK and PILOT_GPU_READY
INPUTS_OK = MODELS_OK = RESULTS_READY = False
run([PY,RUNNER,'prepare'])
PRIVATE=ROOT/'rendered/colab-lipsync'
from IPython.display import display, Image as DisplayImage, Audio, Video
inputs=json.loads((PRIVATE/'inputs/pilot-inputs.json').read_text())
for job in inputs['inputs']:
    print('سجاد' if job['speaker']=='male' else 'آوا', '—',round(job['duration'],2),'ثانیه')
    display(DisplayImage(filename=str(PRIVATE/'inputs'/f"{job['speaker']}-calibration.png")))
    display(Audio(filename=str(PRIVATE/'inputs'/job['playbackAudio'])))
print('آبی: جعبهٔ مدل؛ زرد: ناحیهٔ لب؛ قرمز: خط حفاظت تجهیزات. این‌ها ماسک آزمایشی‌اند، نه تأیید کیفیت لب‌خوانی.')
INPUTS_OK=True


In [ ]:
#@title ۴) دریافت کد و سه مدل رسمی (بدون حساب Hugging Face یا توکن)
assert INPUTS_OK and SETUP_OK and GOOGLE_PROCESSING_CONFIRMED
MODELS_OK = RESULTS_READY = False
MUSE=Path('/content/MuseTalk')
MUSE_PIN='0a89dec45a0192b824e3cf4daf96c239440c5ed8'
if not (MUSE/'.git').is_dir():
    run(['git','clone','--depth','1','--filter=blob:none','--no-checkout','https://github.com/TMElyralab/MuseTalk.git',MUSE])
run(['git','-C',MUSE,'fetch','--depth','1','origin',MUSE_PIN])
run(['git','-C',MUSE,'sparse-checkout','set','musetalk'])
run(['git','-C',MUSE,'checkout','--detach',MUSE_PIN])
run([PY,RUNNER,'models','--musetalk',MUSE])
MODELS_OK=True
print('وزن‌ها آماده شدند؛ نسخه و هش‌ها در model-provenance.json ثبت شدند.')


In [ ]:
#@title ۵) اجرای واقعی مدل و نمایش دو ویدیوی تست
assert MODELS_OK and INPUTS_OK and PILOT_GPU_READY
RESULTS_READY=False
run([PY,RUNNER,'infer','--musetalk',MUSE])
OUT=PRIVATE/'outputs'
result=json.loads((OUT/'pilot-results.json').read_text())
assert len(result['results'])==2
for item in result['results']:
    path=OUT/item['file']
    assert path.is_file() and path.stat().st_size>1000
    print('سجاد' if item['speaker']=='male' else 'آوا', '— نیازمند بررسی لب/چهره')
    display(Video(str(path),embed=True,width=640))
print('کنترل فنی انجام شده؛ هماهنگی لب با فارسی و حفظ شکل لب/ریش باید با دیدن و شنیدن تأیید شود.')
RESULTS_READY=True


In [ ]:
#@title ۶) دریافت ZIP نتیجه برای ارسال به همین گفتگو
assert RESULTS_READY
archive=Path('/content/ahangban-lipsync-pilot.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    # Explicit allowlist: NO standalone WAVs, input faces, weights, secrets or caches.
    for name in ('male-studio-pilot.mp4','female-studio-pilot.mp4','pilot-results.json'):
        z.write(OUT/name,arcname=name)
    z.write(PRIVATE/'model-provenance.json',arcname='model-provenance.json')
from google.colab import files
files.download(str(archive))
print('ZIP را دانلود و در همین گفتگو ارسال کن. پس از دانلود: Runtime → Disconnect and delete runtime.')


## چه چیزی هنوز انجام نشده است؟
- این تست، بازبینی انسانی تلفظ/هماهنگی لب را خودکار انجام نمی‌دهد؛ موفقیت فنی تضمین کیفیت نیست.
- حرکت سر/بدن، پلک و زاویهٔ سه‌رخ ساخته نمی‌شوند؛ فعلاً فقط لب روی همان تصویر ثابت ترکیب می‌شود.
- کل اپیزود رندر یا در یوتیوب منتشر نمی‌شود. ابتدا هر دو تست باید قبول شوند.
- اگر GPU/دانلود/نصب/کیفیت مشکل داشت، پیام خطا یا دو ویدیو را برگردان؛ هیچ سرویس پولی خودکار جایگزین نمی‌شود.

### مجوز و منابع
MuseTalk V1.5: Tencent Music Entertainment Group، MIT؛ بالادست استفادهٔ تجاری از مدل اصلی را مجاز می‌داند. VAE و Whisper طبق مجوزهایشان (MIT) استفاده می‌شوند. داده‌های تست بالادست وارد این آزمایش نشده‌اند. در این مسیر فقط UNet/VAE/Whisper استفاده می‌شوند، نه تشخیص ژست/چهره یا مدل face parsing.

[مجوز بالادست](https://github.com/TMElyralab/MuseTalk/blob/0a89dec45a0192b824e3cf4daf96c239440c5ed8/LICENSE) · [راهنمای پروژه](https://github.com/Masoumiofficial/ahangban-youtube/blob/arena/d4ba4e0b-ahangban-youtube/content/005-cinematic-presenters/colab/README.fa.md)
